# QAIRT 2.42 Qwen2.5-1.5B Genie/HTP build

Build the exact locked official Q4_K_M GGUF into an SM8850 / HTP-v81 `LLMContainer`. This is a high-RAM, offline compilation job. It does not connect to a phone and does not prove NPU execution. Select an Ubuntu 22.04 x86_64 high-RAM Colab runtime and ensure Google Drive and `/content` each have at least 24 GB free.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess
EXPECTED = {
    'qairt242_colab_qwen25_genie.py': 'f5f602276289ec289d9b90595273739f82448e7fa1b45e6b259dc89963c8c5bd',
    'qairt242_qwen25_genie_spec.json': '8b25f22513a78aa9cea7898720ead903e41a77ef4475506450e17fa816c4565a',
    'qairt242_qwen25_15b_genie_source_lock.json': 'cd8af4ced04ff562fbf9b2916a2d2065dfa092c3b4f8e78693e952eb1aa65e5e',
}
def sha256_bytes(data): return hashlib.sha256(data).hexdigest()
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()
print({'platform': platform.platform(), 'machine': platform.machine()})


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
CONTROL = Path('/content/drive/MyDrive/lmf_qairt242_qwen25_genie_v1/control')
WORKSPACE = Path('/content/drive/MyDrive/lmf_qairt242_qwen25_genie_v1/workspace')
SCRATCH = Path('/content/qairt242_qwen25_scratch')
CONTROL.mkdir(parents=True, exist_ok=True); WORKSPACE.mkdir(parents=True, exist_ok=True)
for name, expected in EXPECTED.items():
    path = CONTROL / name
    if path.exists() and sha256_file(path) != expected: raise RuntimeError(f'control hash drift: {path}')
missing = [name for name in EXPECTED if not (CONTROL / name).is_file()]
if missing:
    print('Upload these exact repository files:', missing)
    uploaded = files.upload()
    for name in missing:
        data = uploaded.get(name)
        if data is None or sha256_bytes(data) != EXPECTED[name]: raise RuntimeError(f'bad control file: {name}')
        temporary = CONTROL / f'.{name}.partial'; temporary.write_bytes(data); temporary.replace(CONTROL / name)
print({'control_verified': True, 'workspace': str(WORKSPACE), 'scratch': str(SCRATCH)})


In [ ]:
release = {}
for line in Path('/etc/os-release').read_text().splitlines():
    if '=' in line: release[line.split('=',1)[0]] = line.split('=',1)[1].strip('\"')
if release.get('VERSION_ID') != '22.04': raise RuntimeError(f'QAIRT 2.42 requires Ubuntu 22.04, got {release.get("VERSION_ID")}')
apt_env = dict(os.environ); apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['apt-get','update','-qq'], check=True, env=apt_env)
subprocess.run(['apt-get','install','-y','--no-install-recommends','python3.10','python3.10-venv','python3-distutils','build-essential','clang-14','cmake','ninja-build','curl','unzip','ca-certificates'], check=True, env=apt_env)
python310 = shutil.which('python3.10')
if not python310: raise RuntimeError('python3.10 unavailable')
subprocess.run([python310, '--version'], check=True); subprocess.run(['clang-14','--version'], check=True)


In [ ]:
runner = CONTROL / 'qairt242_colab_qwen25_genie.py'
spec = CONTROL / 'qairt242_qwen25_genie_spec.json'
source_lock = CONTROL / 'qairt242_qwen25_15b_genie_source_lock.json'
subprocess.run([python310, str(runner), 'plan', '--spec', str(spec), '--source-lock', str(source_lock)], check=True)


Run or resume. The first run downloads the checksum-pinned 1.54 GB QAIRT archive and 1.12 GB GGUF, inspects every GGUF tensor type/name, then uses the durable QAIRT `cache_root`. A disconnect can be resumed by rerunning the notebook. No QAIRT 2.45 or CPU fallback is permitted.


In [ ]:
subprocess.run([python310, str(runner), 'run', '--spec', str(spec), '--source-lock', str(source_lock), '--workspace', str(WORKSPACE), '--scratch', str(SCRATCH)], check=True)


In [ ]:
subprocess.run([python310, str(runner), 'verify', '--spec', str(spec), '--source-lock', str(source_lock), '--workspace', str(WORKSPACE)], check=True)
manifest_path = WORKSPACE / 'stages/deliverable/result_manifest.json'
manifest = json.loads(manifest_path.read_text())
print({'status': manifest['status'], 'container_root': str(WORKSPACE / manifest['container_root']), 'context_files': len([x for x in manifest['container_artifacts'] if x['path'].endswith('/model.bin')]), 'phone_execution_proven': manifest['phone_execution_proven'], 'npu_execution_proven': manifest['npu_execution_proven']})
